# Chapter 5 &mdash; Best Practices: Mnemonic State Names

**Concept 3 of the Chapter 5 decomposition:** *Best Practices: Mnemonic State Names and Documented Transitions*

Name states for what they remember; keep transitions consistent with the names; comment every line.

---

*Run on Colab:* [![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ganeshutah/Jove/blob/master/Chapter5/Concept-Mnemonic-State-Names/Concept-Mnemonic-State-Names.ipynb)
*Or run locally from inside a Jove checkout.*

## 0. Setup

In [ ]:
#~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~
# Run this cell first. It works both on Colab and on your own machine.
#~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~
import sys

try:                       # -- are we on Colab? --
    import google.colab
    OWN_INSTALL = False
except ImportError:
    OWN_INSTALL = True

if OWN_INSTALL:
    # Running from Jove/Chapter<N>/Concept-<Name>/ : reach the Jove root.
    sys.path[0:0] = ['../..', '../../3rdparty',
                     '../../..', '../../../3rdparty',
                     '..', '../3rdparty', '.']
else:
    ! if [ ! -d Jove ]; then git clone -q https://github.com/ganeshutah/Jove Jove; fi
    sys.path.append('./Jove')
    sys.path.append('./Jove/jove')

# -- imports needed by this notebook --
from jove.Def_md2mc      import *
from jove.DotBashers     import *
from jove.Def_DFA        import *
from jove.AnimateDFA     import *
#~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~
print("Jove loaded. Ready.")

## 1. The idea


Two habits separate DFA you can debug from DFA you cannot:

* **name a state for what it remembers**, not `q0, q1, q2`;
* **write a `!!` comment on every transition**, saying why it goes there.

Jove's markdown supports both. `md2mc` reads the leading letters (`I`, `F`, `IF`) for
the machine's structure and ignores the rest of the name, so `IF_even_even` is both
initial and final **and** self-documenting.

The pay-off comes when a test fails: a mnemonic name tells you instantly which
invariant broke.

## 2. Definitions

### The same machine, opaque

In [ ]:
opaque = md2mc('''DFA
IF : 0 -> A
IF : 1 -> B
A  : 0 -> IF
A  : 1 -> C
B  : 0 -> C
B  : 1 -> IF
C  : 0 -> B
C  : 1 -> A
''')

### and mnemonic, with a comment per transition

In [ ]:
clear = md2mc('''DFA
!!  State name records (parity of 0s, parity of 1s)
IF_ev0_ev1 : 0 -> S_od0_ev1   !! saw a 0: zero-parity flips
IF_ev0_ev1 : 1 -> S_ev0_od1   !! saw a 1: one-parity flips
S_od0_ev1  : 0 -> IF_ev0_ev1  !! second 0 restores even
S_od0_ev1  : 1 -> S_od0_od1
S_ev0_od1  : 0 -> S_od0_od1
S_ev0_od1  : 1 -> IF_ev0_ev1  !! second 1 restores even
S_od0_od1  : 0 -> S_ev0_od1
S_od0_od1  : 1 -> S_od0_ev1
''')

## 3. Tests

Same language &mdash; the names cost nothing at runtime.

In [ ]:
print("same language? ", langeq_dfa(opaque, clear))
print("isomorphic?    ", iso_dfa(opaque, clear))
assert langeq_dfa(opaque, clear) and iso_dfa(opaque, clear)

But the names are **checkable invariants**: the state must match the parities seen.

In [ ]:
def parity_pair(s): return (s.count('0') % 2, s.count('1') % 2)
tag = {'IF_ev0_ev1': (0,0), 'S_od0_ev1': (1,0), 'S_ev0_od1': (0,1), 'S_od0_od1': (1,1)}

from itertools import product
for k in range(6):
    for p in product('01', repeat=k):
        s = ''.join(p)
        assert tag[run_dfa(clear, s)] == parity_pair(s), s
print("every state's NAME matches what it actually remembers, on all strings up to length 5")
print("\nThat assertion is only writable because the names mean something.")

With opaque names the same check is impossible to state, so the bug would hide.

In [ ]:
print("states of the opaque machine :", sorted(opaque["Q"]))
print("what does 'C' remember? -- you have to re-derive it every time.")

## 4. Animation

Mnemonic names make the picture readable too.

*(The `display(HTML(...))` line loads the toolbar's font-awesome icons. Keep it last in the cell &mdash; it must be there for the controls to appear.)*

In [ ]:
from jove.AnimateDFA import *
AnimateDFA(clear, FuseEdges=True)
display(HTML('<link rel="stylesheet" href="//stackpath.bootstrapcdn.com/font-awesome/4.7.0/css/font-awesome.min.css"/>'))

## 5. Exercises


1. Rename the states of a DFA you wrote earlier. Did you find a bug?
2. Write the invariant assertion for the `01`-containing DFA of Chapter 4.
3. Why can `md2mc` ignore everything after the leading `I`/`F`?

In [ ]:
# Your work for the exercises above.